In [2]:
# ============================================================
# TEXT SUMMARIZATION APPLICATION - GOOGLE COLAB
# Single Cell Application
# ============================================================

!pip -q install -U transformers sentencepiece gradio

import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ------------------------------------------------------------
# Load Summarization Model
# ------------------------------------------------------------

MODEL_NAME = "facebook/bart-large-cnn"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded successfully!")
print("Running on:", device)


# ------------------------------------------------------------
# Summarization Function
# ------------------------------------------------------------

def summarize_text(text):

    if not text.strip():
        return "Please enter some text."

    words = text.split()

    if len(words) < 30:
        return "Please enter at least 30 words for better summarization."

    # Limit input size
    text = " ".join(words[:700])

    # Convert text into tokens
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Generate summary
    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_length=130,
            min_length=30,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )

    # Convert tokens back to text
    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary


# ------------------------------------------------------------
# Gradio Application
# ------------------------------------------------------------

with gr.Blocks(title="Text Summarization Application") as app:

    gr.Markdown(
        """
        # 📝 Text Summarization Application

        ### NLP-Based Automatic Text Summarizer

        Enter a long paragraph or article and click **Summarize Text**
        to generate a short and meaningful summary.
        """
    )

    input_text = gr.Textbox(
        label="Enter Your Text",
        placeholder="Paste your article or paragraph here...",
        lines=12
    )

    summarize_button = gr.Button(
        "✨ Summarize Text",
        variant="primary"
    )

    output_text = gr.Textbox(
        label="Generated Summary",
        lines=8
    )

    summarize_button.click(
        fn=summarize_text,
        inputs=input_text,
        outputs=output_text
    )

    gr.Markdown(
        """
        **Technology Used:** Python | NLP | Transformers | BART | Gradio
        """
    )


# ------------------------------------------------------------
# Start Application
# ------------------------------------------------------------

app.launch(share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.3 MB/s eta 0:00:00
Loading model...


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model loaded successfully!
Running on: cpu
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2893b46a1b05e784de.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
